In [1]:
import sys
sys.path.append("..")

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import itertools
import tqdm
from src.algorithm import find_partitions_greedy, find_partitions_optimal
from src.metric import approximation_ratio
from src.subroutine import evaluate_system

In [3]:
np.random.seed(0)

In [4]:
x_min, x_max, x_disc = 0., 1., 1e-4
X = np.arange(x_min, x_max+x_disc, x_disc).round(4)
N_THRESHOLDS = 3

In [5]:
def generate_threshold_grid(n_components, step=0.02):
    ticks = np.round(np.arange(0, 1 + step, step), 10)
    return [np.array(combo) for combo in itertools.combinations(ticks, n_components)]

In [6]:
np.random.seed(0)

def sweep(threshold_grid):
    d = {"thresholds": [], "priors": [], "c": [], "t*": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r_mult": [], "r_add": []}
    c = 1.0
    priors = np.random.dirichlet([2] * N_THRESHOLDS)
    # threshold_true = np.min(thresholds)
    # threshold_true = np.median(thresholds)
    for thresholds in tqdm.tqdm(threshold_grid): 
        threshold_true = np.min(thresholds)
        partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
        partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)
        loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
        loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
        r_mult  = approximation_ratio(loss_opt, loss_greedy, rtype="mult")
        r_add  = approximation_ratio(loss_opt, loss_greedy, rtype="add")
        if not np.isnan(r_mult):
            d["thresholds"].append(thresholds)
            d["priors"].append(priors)
            d["c"].append(c)
            d["t*"].append(threshold_true)
            d["partition_opt"].append(partition_opt)
            d["partition_greedy"].append(partition_greedy)
            d["loss_opt"].append(loss_opt.item())
            d["loss_greedy"].append(loss_greedy.item())
            d["r_mult"].append(r_mult.item())
            d["r_add"].append(r_add.item())

    return d

In [7]:
threshold_grid = generate_threshold_grid(N_THRESHOLDS, step=0.02)
d = sweep(threshold_grid)

100%|██████████| 20825/20825 [02:17<00:00, 151.02it/s]


In [9]:
df = pd.DataFrame(d)
i = np.argmax(d["r_mult"])
df.iloc[[i]]

,thresholds,priors,c,t*,partition_opt,partition_greedy,loss_opt,loss_greedy,r_mult,r_add
16838,"[0.48, 0.58, 1.0]","[0.4013317620800753, 0.174820642593116, 0.4238...",1.0,0.48,"[[1], [0, 2]]","[[0, 1, 2]]",0.279238,0.479852,1.718435,0.200614


In [15]:
fig = px.scatter(df, x="t*", y="r_mult", color="loss_greedy")
fig.update_layout(
    width=550, height=450,
    font=dict(family="iosevka"),
    yaxis=dict(title="Approximation Ratio"),
    margin=dict(t=25,b=25,l=25,r=25)
    )
# fig.update_traces(marker=dict(color="rgba(37, 99, 235, 1)"))